<a href="https://colab.research.google.com/github/LikithaC17/GramaSetu-AI/blob/main/GramaSetu_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**GramaSetu AI – Complete Rural Commerce Ecosystem**
GramaSetu AI is an AI-powered rural commerce ecosystem that connects farmers, Kirana stores, artisans, Women's SHGs, and logistics providers. Powered by a fine-tuned LLM, it supports group buying, inventory management, transportation optimization, market access, financial management, and rural employment to promote inclusive rural development.

Import Libraries

In [50]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("Running on CPU")

PyTorch version: 2.14.1+cu130
GPU available: True
GPU: Tesla T4
GPU memory: 14.56 GB


Install Libraries

In [51]:
!pip install -U git+https://github.com/huggingface/transformers.git accelerate peft safetensors

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-4g324cvy
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-4g324cvy
  Resolved https://github.com/huggingface/transformers.git to commit f6b7cbef762d0b5997a2dcab0a6ea33f28f68753
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [52]:
import torch
import transformers
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
Transformers: 5.25.0.dev0
PEFT: 0.22.0
Accelerate: 1.16.0
GPU: Tesla T4


Finding File

In [53]:
import os

for file in os.listdir("/content"):
    print(file)

.config
gramasetu_full.tar
adaption_qwen3_5_2b_gramasetu_rural_commerce_2892bb55.tgz
drive
gramasetu_model
sample_data


Printing File

In [54]:
import os

print(os.listdir("/content"))

['.config', 'gramasetu_full.tar', 'adaption_qwen3_5_2b_gramasetu_rural_commerce_2892bb55.tgz', 'drive', 'gramasetu_model', 'sample_data']


Install zstandard

In [55]:
!pip install zstandard

Import zstandard

In [56]:
import os
import zstandard as zstd

path = "/content/adaption_qwen3_5_2b_gramasetu_rural_commerce_2892bb55.tgz"

print("Archive exists:", os.path.exists(path))
print("Archive size:", round(os.path.getsize(path) / (1024**2), 2), "MB")

with open(path, "rb") as f:
    print("File signature:", f.read(16).hex())

print("Zstandard check:")

try:
    with open(path, "rb") as f:
        reader = zstd.ZstdDecompressor().stream_reader(f)
        data = reader.read()
        reader.close()
    print("Decompression successful:", len(data), "bytes")
except Exception as e:
    print("Decompression failed:", repr(e))

Archive exists: True
Archive size: 157.67 MB
File signature: 28b52ffd04582c860a0eb09569502d10
Zstandard check:
Decompression successful: 194826240 bytes


Print File

In [59]:
import os

path = "/content/adaption_qwen3_5_2b_gramasetu_rural_commerce_2892bb55.tgz"

print("File exists:", os.path.exists(path))
if os.path.exists(path):
    print("File size:", round(os.path.getsize(path) / (1024**2), 2), "MB")

File exists: True
File size: 157.67 MB


Extraction of Dataset

In [60]:
import os
import zstandard as zstd
import tarfile

source = "/content/adaption_qwen3_5_2b_gramasetu_rural_commerce_2892bb55.tgz"
output = "/content/gramasetu_full.tar"
extract_path = "/content/gramasetu_model"

with open(source, "rb") as compressed:
    with open(output, "wb") as decompressed:
        zstd.ZstdDecompressor().copy_stream(compressed, decompressed)

os.makedirs(extract_path, exist_ok=True)

with tarfile.open(output, "r:") as archive:
    archive.extractall(extract_path, filter="data")

print("Extraction completed!")

for root, dirs, files in os.walk(extract_path):
    for name in files:
        print(name, os.path.getsize(os.path.join(root, name)))

Extraction completed!
processor_config.json 1191
adapter_config.json 10254
README.md 5101
tokenizer.json 19989325
config.json 2734
chat_template.jinja 7650
tokenizer_config.json 1192
preprocessor_config.json 443
adapter_model.safetensors 174616272
special_tokens_map.json 325
trainer_state.json 176362


Extract Contents in File  

In [61]:
import json

folder = "/content/gramasetu_model"

for filename in ["adapter_config.json", "config.json"]:
    print("\n", filename)
    with open(f"{folder}/{filename}", "r") as f:
        data = json.load(f)

    if filename == "adapter_config.json":
        print("Base model:", data.get("base_model_name_or_path"))
        print("LoRA rank:", data.get("r"))
        print("LoRA alpha:", data.get("lora_alpha"))
        print("Target modules:", data.get("target_modules"))
    else:
        print("Model architecture:", data.get("model_type"))
        print("Architectures:", data.get("architectures"))


 adapter_config.json
Base model: togethercomputer/Qwen3.5-2B
LoRA rank: 64
LoRA alpha: 128
Target modules: ['gate_proj', 'up_proj', 'o_proj', 'down_proj', 'v_proj', 'q_proj', 'k_proj']

 config.json
Model architecture: qwen3_5
Architectures: ['Qwen3_5ForConditionalGeneration']


Import Model-1 from Hugging Face

In [62]:
from huggingface_hub import HfApi

model_id = "togethercomputer/Qwen3.5-2B"

try:
    info = HfApi().model_info(model_id)
    print("Repository found:", info.id)
    print("Repository status: Accessible")
except Exception as e:
    print("Repository check failed:")
    print(type(e).__name__, str(e))

Repository check failed:
RepositoryNotFoundError 401 Client Error. (Request ID: Root=1-6b09856f-005e62df56732ae847a2333d;51d93030-28e8-4454-a1ba-09717e4d95fe)

Repository Not Found for url: https://huggingface.co/api/models/togethercomputer/Qwen3.5-2B.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.


Import Model-2 from Hugging Face

In [63]:
from huggingface_hub import HfApi

model_id = "Qwen/Qwen3.5-2B"

try:
    info = HfApi().model_info(model_id)
    print("Repository:", info.id)
    print("Status: Accessible")
    print("Architecture:", info.config.get("model_type"))
except Exception as e:
    print(type(e).__name__, str(e))

Repository: Qwen/Qwen3.5-2B
Status: Accessible
Architecture: qwen3_5


In [64]:
import json

path = "/content/gramasetu_model/adapter_config.json"

with open(path, "r") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "alpha_pattern": {},
  "auto_mapping": null,
  "base_model_name_or_path": "togethercomputer/Qwen3.5-2B",
  "bias": "none",
  "corda_config": null,
  "eva_config": null,
  "exclude_modules": [
    "model.visual.blocks.5.attn.proj",
    "model.visual.blocks.6.mlp",
    "model.visual.blocks.14.mlp.linear_fc1",
    "model.visual.blocks.15.attn",
    "model.visual.blocks.21.norm1",
    "model.visual.blocks.2.mlp.linear_fc2",
    "model.visual.blocks.14.norm2",
    "model.visual.blocks.20.mlp",
    "model.visual.blocks.14",
    "model.visual.blocks.6.attn",
    "model.visual.blocks.17.mlp.linear_fc1",
    "model.visual.blocks.13.attn.qkv",
    "model.visual.blocks.3.attn.proj",
    "model.visual.blocks",
    "model.visual.blocks.0.mlp.linear_fc1",
    "model.visual.blocks.7.mlp.linear_fc2",
    "model.visual.blocks.9.mlp.act_fn",
    "model.visual.blocks.10.mlp.linear_fc1",
    "model.visual.blocks.21.mlp.linear_fc2",
    "model.visual.blocks.5.mlp.linear_fc2",
    "model.visual.blocks.1

Extract Content

In [65]:
import json

path = "/content/gramasetu_model/adapter_config.json"

with open(path, "r") as f:
    config = json.load(f)

config["base_model_name_or_path"] = "Qwen/Qwen3.5-2B"

with open(path, "w") as f:
    json.dump(config, f, indent=2)

print("Updated base model:", config["base_model_name_or_path"])
print("LoRA rank:", config["r"])
print("LoRA alpha:", config["lora_alpha"])

Updated base model: Qwen/Qwen3.5-2B
LoRA rank: 64
LoRA alpha: 128


Import torch

In [66]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM
from peft import PeftModel

print("PyTorch:", torch.__version__)
print("Multimodal model class:", AutoModelForMultimodalLM)
print("PEFT:", PeftModel)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
Multimodal model class: <class 'transformers.models.auto.modeling_auto.AutoModelForMultimodalLM'>
PEFT: <class 'peft.peft_model.PeftModel'>
GPU available: True


Import torch with Base-Model

In [67]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

BASE_MODEL_ID = "Qwen/Qwen3.5-2B"

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

base_model = AutoModelForMultimodalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Base model loaded successfully!")
print("Model:", BASE_MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", base_model.device)

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Base model loaded successfully!
Model: Qwen/Qwen3.5-2B
GPU: Tesla T4
Model device: cuda:0


From peft Import PeftModel

In [68]:
from peft import PeftModel

ADAPTER_PATH = "/content/gramasetu_model"

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False
)

model.eval()

print("GramaSetu AI adapter loaded successfully!")
print("Adapter:", ADAPTER_PATH)
print("Model device:", model.device)

GramaSetu AI adapter loaded successfully!
Adapter: /content/gramasetu_model
Model device: cuda:0


Round 1

Test 1-Group Buying and Bulk Purchase Calculation

In [69]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "A group of 20 farmers wants to purchase 500 kg of fertilizer. The normal price is ₹50 per kg, but a bulk supplier offers ₹45 per kg. Calculate the total cost at both prices and the total savings. Explain how GramaSetu AI can help coordinate this group purchase."
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    key: value.to(model.device) if isinstance(value, torch.Tensor) else value
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("GRAMASETU AI RESPONSE:\n")
print(response)

GRAMASETU AI RESPONSE:

### Financial Analysis: Bulk Purchase vs. Normal Price

Here is the step-by-step calculation for the total cost and savings based on the provided data.

**1. Calculate Total Cost at Normal Price**
*   **Quantity:** 500 kg
*   **Price per kg:** ₹50
*   **Calculation:** 500 kg × ₹50 = **₹25,000**

**2. Calculate Total Cost at Bulk Price**
*   **Quantity:** 500 kg
*   **Price per kg:** ₹45
*   **Calculation:** 500 kg × ₹45 = **₹22,500**

**3. Calculate Total Savings**
*   **Calculation:** ₹25,000 (Normal) - ₹22,500 (Bulk) = **₹2,500**
*   **Savings per kg:** ₹50 - ₹45 = ₹5 per kg
*   **Total Savings:** 500 kg × ₹5 = **₹2,500**

**Summary:** By coordinating a group purchase, the farmers save **₹2,500** (a 10% reduction in expenditure).

---

### How GramaSetu AI Can Help Coordinate This Group Purchase

GramaSetu AI is designed to bridge


Test 2-Kirana Store Inventory Management

In [70]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### Financial Analysis: Bulk Purchase vs. Normal Price

Here is the step-by-step calculation for the total cost and savings based on the provided data.

**1. Calculate Total Cost at Normal Price**
*   **Quantity:** 500 kg
*   **Price per kg:** ₹50
*   **Calculation:** 500 kg × ₹50 = **₹25,000**

**2. Calculate Total Cost at Bulk Price**
*   **Quantity:** 500 kg
*   **Price per kg:** ₹45
*   **Calculation:** 500 kg × ₹45 = **₹22,500**

**3. Calculate Total Savings**
*   **Calculation:** ₹25,000 (Normal) - ₹22,500 (Bulk) = **₹2,500**
*   **Savings per kg:** ₹50 - ₹45 = ₹5 per kg
*   **Total Savings:** 500 kg × ₹5 = **₹2,500**

**Summary:** By coordinating a group purchase, the farmers save **₹2,500** (a 10% reduction in expenditure).

---

### How GramaSetu AI Can Help Coordinate This Group Purchase

GramaSetu AI is designed to bridge the gap between individual smallholder farmers and large-scale supply chains. In this scenario, it can assist the group of 20 farmers in the following ways

Test 3- Rural Transportation Cost Optimization

In [71]:
test_prompt = """
A village Kirana shop sells 40 packets of rice per day.
Currently, it has 120 packets in stock.
Its supplier takes 4 days to deliver new stock.
The shopkeeper wants to maintain 3 extra days of safety stock.

Calculate:
1. How many days will the current stock last?
2. What is the reorder point?
3. When should the shopkeeper place the next order?

Explain how GramaSetu AI can help prevent stockouts and reduce excess inventory.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Calculation of Days Until Stockout
To determine how long the current stock will last, divide the total inventory by the daily consumption rate.
*   **Current Stock:** 120 packets
*   **Daily Consumption:** 40 packets
*   **Calculation:** 120 ÷ 40 = 3 days

**Answer:** The current stock will last for **3 days**.

---

### 2. Calculation of Reorder Point (ROP)
The Reorder Point is the specific inventory level at which a new order must be placed to avoid stockouts. It is calculated by adding the safety stock to the current stock.
*   **Current Stock:** 120 packets
*   **Safety Stock:** 3 packets (3 extra days × 40 packets/day)
*   **Calculation:** 120 + 3 = 123 packets

**Answer:** The reorder point is **123 packets**.

---

### 3. Timing for the Next Order
The shopkeeper should place the next order immediately when the inventory level drops to or below the reorder point.
*   **Trigger:** Once the stock reaches 123 packets.
*   **Action:** Place an order for 123 packets (assuming t

Test 4-Farmers' Market Access and Profit Calculation

In [72]:
test_prompt = """
A rural farmers group needs to transport 1,200 kg of vegetables to a nearby market.

Transport options:
Option A: Small vehicle, capacity 400 kg, cost ₹1,500 per trip.
Option B: Large vehicle, capacity 1,200 kg, cost ₹3,800 per trip.

Calculate:
1. Number of trips required for each option.
2. Total transportation cost for each option.
3. Which option is cheaper?
4. Total savings using the cheaper option.

Explain how GramaSetu AI can coordinate transportation for the farmers and reduce logistics costs.
Show all calculations clearly.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Number of Trips Required

To determine the number of trips, we divide the total weight of the vegetables by the capacity of each vehicle option.

*   **Option A (Small Vehicle):**
    *   Total Weight: 1,200 kg
    *   Capacity: 400 kg
    *   Calculation: $1,200 \text{ kg} \div 400 \text{ kg} = 3$ trips
*   **Option B (Large Vehicle):**
    *   Total Weight: 1,200 kg
    *   Capacity: 1,200 kg
    *   Calculation: $1,200 \text{ kg} \div 1,200 \text{ kg} = 1$ trip

### 2. Total Transportation Cost

Next, we calculate the total cost for each option by multiplying the number of trips by the cost per trip.

*   **Option A (Small Vehicle):**
    *   Trips: 3
    *   Cost per Trip: ₹1,500
    *   Calculation: $3 \text{ trips} \times ₹1,500 = ₹4,500$
*   **Option B (Large Vehicle):**
    *   Trips: 1
    *   Cost per Trip: ₹3,800
    *   Calculation: $1 \text{ trip} \times ₹3,800 = ₹3,800$

### 3. Which Option is Cheaper?

Comparing the total costs:
*   Option A: ₹4,500
*   Option B: 

Test 5- Artisan Marketplace Revenue Comparison

In [73]:
test_prompt = """
A farmer harvests 800 kg of tomatoes.

Selling options:
Option A: Sell to a local trader at ₹18 per kg with no transportation cost.
Option B: Sell directly in a town market at ₹25 per kg, but transportation costs ₹2,000 and market fees are ₹500.

Calculate:
1. Total revenue under Option A.
2. Total revenue under Option B before expenses.
3. Net earnings under both options after expenses.
4. Which option is more profitable?
5. How much additional profit does the farmer earn using the better option?

Explain how GramaSetu AI can help farmers access markets, compare buyers, and reduce dependence on intermediaries.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Total Revenue Calculation

**Option A: Local Trader**
*   **Formula:** Weight × Selling Price
*   **Calculation:** 800 kg × ₹18/kg
*   **Total Revenue:** **₹14,400**

**Option B: Town Market**
*   **Formula:** Weight × Selling Price
*   **Calculation:** 800 kg × ₹25/kg
*   **Total Revenue:** **₹20,000**

---

### 2. Net Earnings Calculation

To find the net earnings, we subtract the specified expenses from the total revenue.

**Option A: Local Trader**
*   **Total Revenue:** ₹14,400
*   **Transportation Cost:** ₹0
*   **Market Fees:** ₹0
*   **Net Earnings:** ₹14,400 - ₹0 - ₹0 = **₹14,400**

**Option B: Town Market**
*   **Total Revenue:** ₹20,000
*   **Transportation Cost:** ₹2,000
*   **Market Fees:** ₹500
*   **Total Expenses:** ₹2,000 + ₹500 = ₹2,500
*   **Net Earnings:** ₹20,000 - ₹2,500 = **₹17,500**

---

### 3. Profitability Comparison

**Which option is more profitable?**
**Option B (Town Market)** is more profitable. While Option A offers a higher per-kg price, the sig

Test 5- Artisan Marketplace Revenue Comparison

In [74]:
test_prompt = """
A rural artisan group produces 150 handmade baskets.

They have two selling options:

Option A: Sell to a local wholesaler at ₹120 per basket with no platform commission.
Option B: Sell directly through an online marketplace at ₹180 per basket. The marketplace charges a 10% commission on the selling price per basket. Packaging and shipping cost ₹1,500 in total.

Calculate:
1. Total revenue under Option A.
2. Gross revenue under Option B.
3. Total marketplace commission.
4. Net earnings under both options after applicable expenses.
5. Which option is more profitable?
6. Additional earnings from choosing the better option.

Explain how GramaSetu AI can help rural artisans reach customers, price their products, and manage online orders.
Show all calculations clearly.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Total Revenue under Option A
To determine the total revenue under Option A, we multiply the number of baskets produced by the wholesale price per basket.
*   **Formula:** Revenue = Quantity × Selling Price
*   **Calculation:** 150 baskets × ₹120 per basket
*   **Total Revenue:** **₹18,000**

### 2. Gross Revenue under Option B
Gross revenue represents the total amount received from the customer before deducting marketplace fees.
*   **Formula:** Gross Revenue = Quantity × Selling Price
*   **Calculation:** 150 baskets × ₹180 per basket
*   **Total Gross Revenue:** **₹27,000**

### 3. Total Marketplace Commission
The marketplace charges a 10% commission on the selling price per basket.
*   **Formula:** Commission = Gross Revenue × Commission Rate
*   **Calculation:** ₹27,000 × 10%
*   **Total Commission:** **₹2,700**

### 4. Net Earnings under Both Options
Net earnings are calculated by subtracting total expenses from the gross revenue.
*   **Common Expenses:** Packaging and ship

Test 6- Women's SHG Financial Management

In [75]:
test_prompt = """
A Women's Self-Help Group (SHG) has 12 members who produce handmade cloth bags.

Each member produces 25 bags per month.
The selling price is ₹80 per bag.
The total monthly expenses for raw materials, packaging, and transportation are ₹8,000.

Calculate:
1. Total bags produced per month.
2. Total monthly sales revenue.
3. Net monthly profit after expenses.
4. Equal profit share per member.
5. Monthly income per member if each member also receives an equal share of the profit.

Assume all bags are sold and expenses are shared equally. Clearly distinguish sales revenue, profit, and total income per member.

Explain how GramaSetu AI can help SHGs manage production, orders, payments, and market access.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Financial Calculations for the Women's SHG

Here is the step-by-step breakdown of the financial performance based on the provided data:

*   **Total bags produced per month:** 12 members × 25 bags/member = **300 bags**
*   **Total monthly sales revenue:** 300 bags × ₹80/bag = **₹24,000**
*   **Net monthly profit after expenses:** ₹24,000 (Revenue) - ₹8,000 (Expenses) = **₹16,000**
*   **Equal profit share per member:** ₹16,000 ÷ 12 members = **₹1,333.33**
*   **Monthly income per member (Profit share):** **₹1,333.33**

*(Note: The total monthly income per member includes both the profit share and the equal share of the fixed expenses. If calculating only the profit share, it is ₹1,333.33. If calculating total income per member, it is ₹1,333.33 + ₹666.67 = ₹2,000.)*

---

### 2. Detailed Explanations for GramaSetu AI

GramaSetu AI is a digital platform designed to bridge the gap between rural artisans and urban markets. For a Self-Help Group (SHG) like this, the platform acts as 

Test 7-Rural Youth Employment Analytics

In [76]:
test_prompt = """
A rural employment initiative trains 40 unemployed young people.

After training:
- 30 successfully complete the program.
- 24 of those who complete training get jobs.
- Each employed person earns ₹12,000 per month.

Calculate:
1. Training completion rate as a percentage of enrolled participants.
2. Employment rate among those who completed training.
3. Overall employment rate among all enrolled participants.
4. Total monthly income generated by the employed youth.
5. Number of trained participants who remain unemployed.

Explain how GramaSetu AI can connect rural youth with skill training, local businesses, job opportunities, and employers.

Show all calculations and distinguish the different employment rates.
"""

messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": test_prompt}]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

### 1. Calculations and Explanations

Here is the step-by-step breakdown of the employment metrics for the initiative.

**Given Data:**
*   **Total Enrolled Participants:** 40
*   **Successfully Completed Training:** 30
*   **Jobs Obtained by Completed Participants:** 24
*   **Monthly Income per Employed Youth:** ₹12,000

**Step 1: Training Completion Rate**
This measures the success of the training curriculum itself.
*   **Formula:** (Number of Completed Participants / Total Enrolled Participants) × 100
*   **Calculation:** (30 / 40) × 100 = **75%**

**Step 2: Employment Rate Among Completed Participants**
This measures the effectiveness of the placement services for those who finished the training.
*   **Formula:** (Number of Jobs Obtained by Completed / Number of Completed Participants) × 100
*   **Calculation:** (24 / 30) × 100 = **80%**

**Step 3: Overall Employment Rate Among Enrolled Participants**
This measures the total success rate of the initiative from the start to finish.


In [77]:
import shutil
import os

source = "/content/gramasetu_model"
destination = "/content/drive/MyDrive/GramaSetu_AI/first_trained_model"

os.makedirs(os.path.dirname(destination), exist_ok=True)

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Backup completed!")
print("Location:", destination)

Backup completed!
Location: /content/drive/MyDrive/GramaSetu_AI/first_trained_model


Round 2

Test 1-Complex Kirana Inventory Calculation

In [79]:

test_prompt = """A rural Kirana store sells 35 packets of rice per day. The current stock is 180 packets. The supplier takes 5 days to deliver new stock. The shopkeeper wants to maintain a safety stock of 4 days of average sales.

Calculate:
1. Average daily demand.
2. Lead-time demand.
3. Safety stock in packets.
4. Reorder point.
5. Whether the shopkeeper should reorder immediately.
6. How many packets should be ordered if the target stock after replenishment is 15 days of average demand.

Show every calculation with the correct units. Do not confuse days with packets. State any assumptions clearly."""

messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


### Assumptions
1. **Constant Demand:** The daily sales rate of 35 packets is constant and does not fluctuate significantly.
2. **No Stockouts:** The shopkeeper does not lose sales due to stockouts during the 5-day lead time.
3. **Lead Time Consistency:** The supplier's delivery time remains consistently 5 days.
4. **Safety Stock Definition:** Safety stock is calculated based on the average daily demand, not the peak demand.

---

### 1. Average Daily Demand
This represents the steady rate at which stock is consumed.
$$ \text{Average Daily Demand} = \frac{\text{Sales per Day}}{1} $$
$$ \text{Average Daily Demand} = \frac{35 \text{ packets}}{1} = \mathbf{35 \text{ packets/day}} $$

### 2. Lead-time Demand
This represents the total amount of stock required from the supplier during the time it takes to receive the new stock.
$$ \text{Lead-time Demand} = \text{Average Daily Demand} \times \text{Lead-time (Days)} $$
$$ \text{Lead-time Demand} = 35 \text{ packets/day} \times 5 \text{ days} =

In [80]:

test_prompt = """Continue the previous inventory calculation.

The shopkeeper wants to have 15 days of average demand available immediately after the supplier's delivery arrives.

Current stock is 180 packets. Expected consumption during the 5-day delivery period is 175 packets.

Calculate:
1. Expected stock when delivery arrives.
2. Target stock after delivery.
3. Exact order quantity required.

Explain the calculation and do not subtract the lead-time demand twice."""

messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=400, do_sample=False)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


### Inventory Calculation Breakdown

To ensure the shopkeeper has a 15-day supply of stock available immediately after the supplier's delivery arrives, we must account for the lead-time demand and the desired on-hand inventory.

**1. Expected Stock When Delivery Arrives**
This represents the current inventory plus the amount ordered to cover the lead-time demand.
*   **Current Stock:** 180 packets
*   **Lead-Time Demand:** 175 packets (Expected consumption during the 5-day delivery period)
*   **Calculation:** 180 + 175 = 355 packets
*   **Result:** 355 packets

**2. Target Stock After Delivery**
This is the total inventory level required to maintain the 15-day average demand buffer.
*   **Average Daily Demand:** 175 packets / 5 days = 35 packets/day
*   **Lead-Time Duration:** 15 days
*   **Calculation:** 35 packets/day × 15 days = 525 packets
*   **Result:** 525 packets

**3. Exact Order Quantity Required**
This is the difference between the target stock and the current stock.
*   **

Test 2-Kannada Language – Group Buying Calculation

In [81]:

test_prompt = """ಒಂದು ಹಳ್ಳಿಯಲ್ಲಿ 20 ರೈತರು ಒಟ್ಟಾಗಿ 500 ಕೆ.ಜಿ. ರಸಗೊಬ್ಬರವನ್ನು ಖರೀದಿಸಲು ಬಯಸುತ್ತಾರೆ. ಸಾಮಾನ್ಯ ಬೆಲೆ ಪ್ರತಿ ಕೆ.ಜಿಗೆ ₹50. ಒಟ್ಟಾಗಿ ದೊಡ್ಡ ಪ್ರಮಾಣದಲ್ಲಿ ಖರೀದಿಸಿದರೆ ಪ್ರತಿ ಕೆ.ಜಿಗೆ ₹45 ಸಿಗುತ್ತದೆ.

ಲೆಕ್ಕಾಚಾರ ಮಾಡಿ:
1. ಸಾಮಾನ್ಯ ಬೆಲೆಯಲ್ಲಿ ಒಟ್ಟು ವೆಚ್ಚ ಎಷ್ಟು?
2. ಒಟ್ಟಾಗಿ ಖರೀದಿಸಿದಾಗ ಒಟ್ಟು ವೆಚ್ಚ ಎಷ್ಟು?
3. ರೈತರಿಗೆ ಒಟ್ಟು ಎಷ್ಟು ಉಳಿತಾಯವಾಗುತ್ತದೆ?
4. ಪ್ರತಿ ರೈತನಿಗೆ ಸಮಾನವಾಗಿ ಹಂಚಿದರೆ ಎಷ್ಟು ಉಳಿತಾಯವಾಗುತ್ತದೆ?

ಹಂತ ಹಂತವಾಗಿ ಕನ್ನಡದಲ್ಲಿ ಉತ್ತರಿಸಿ."""

messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


ನೀವು ಕೊಟ್ಟಿರುವ ಲೆಕ್ಕಾಚಾರವನ್ನು ಹಂತ ಹಂತವಾಗಿ ಕೆಳಗಿನಂತಿರಿಸಲಾಗಿದೆ:

**1. ಸಾಮಾನ್ಯ ಬೆಲೆಯಲ್ಲಿ ಒಟ್ಟು ವೆಚ್ಚ ಎಷ್ಟು?**
ರೈತರ ಸಂಘದ ಸದಸ್ಯರ ಸಂಖ್ಯೆ 20 ಮತ್ತು ಸಾಮಾನ್ಯ ಬೆಲೆ 50 ರಸಗೊಬ್ಬರ ಕೆ.ಜಿಗಿ.
* ಒಟ್ಟು ವೆಚ್ಚ = 20 ರೈತರು × 50 ಕೆ.ಜಿ. × 50 ರಸಗೊಬ್ಬರ
* ಒಟ್ಟು ವೆಚ್ಚ = 1,000 × 50
* ಒಟ್ಟು ವೆಚ್ಚ = **₹50,000**

**2. ಒಟ್ಟಾಗಿ ಖರೀದಿಸಿದಾಗ ಒಟ್ಟು ವೆಚ್ಚ ಎಷ್ಟು?**
ರೈತರ ಸಂಘದ ಸದಸ್ಯರ ಸಂಖ್ಯೆ 20 ಮತ್ತು ಒಟ್ಟಾದ ಪ್ರಮಾಣ 500 ಕೆ.ಜಿ.
* ಒಟ್ಟು ವೆಚ್ಚ = 20 ರೈತರು × 500 ಕೆ.ಜಿ. × 45 ರಸಗೊಬ್ಬರ
* ಒಟ್ಟು ವೆಚ್ಚ = 10,000 × 45
* ಒಟ್ಟು ವೆಚ್ಚ = **₹45,000**

**3. ರೈತರಿಗೆ ಒಟ್ಟು ಎಷ್ಟು ಉಳಿತಾಯವಾಗುತ್ತದೆ?**
ಸಾಮಾನ್ಯ ಬೆಲೆಯಲ್ಲಿ ಒಟ್ಟು ವೆಚ್ಚ ಮತ್ತು ಒಟ್ಟಾದ ಖರೀದಿ ವೆಚ್ಚದ ನಡುವಿನ ವ್ಯತ್ಯಾಸವು ಉಳಿತಾಯವಾಗುತ್ತದೆ.
* ಉಳಿತಾಯ = ₹50,000 - ₹45,000
* ಉಳಿತಾಯ = **₹5,000**

**4. ಪ್ರತಿ ರೈತನಿಗೆ ಸಮಾನವಾಗಿ ಹಂಚಿದರೆ ಎಷ್ಟು ಉಳಿತಾಯವಾಗುತ್ತದೆ?**
ಉಳಿತಾಯವು 20 ರೈತರ ಸಂಘದ ಸದಸ್ಯರ ನಡುವೆ ಸಮಾನವಾಗಿ ಹಂಚಿಕೆಯಾಗುತ್ತದೆ.
* ಪ್ರತಿ ರೈತನ ಉಳಿತಾಯ = ಒಟ್ಟು ಉಳಿತಾಯ ÷ ರೈತರ ಸಂಖ್ಯೆ
* ಪ್ರತಿ ರ


Test 3 – Missing Information & Hallucination Evaluation

In [82]:

test_prompt = """A farmer wants to transport 800 kg of tomatoes to a nearby city market. The farmer knows the selling price at the city market is ₹25 per kg, but does not know the transportation cost or market fees.

Calculate the farmer's exact net profit.

If information is missing, identify exactly what is needed. Do not invent or assume transportation costs, market fees, or purchase prices. Explain what can and cannot be calculated."""

messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=500, do_sample=False)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


To calculate the exact net profit, all necessary financial variables must be known. Based on the information provided, the exact net profit cannot be determined.

**Missing Information**
To calculate net profit, the following data is required:
1. **Purchase Price:** The cost per kilogram of tomatoes at the farm gate.
2. **Transportation Cost:** The total expense for moving the tomatoes from the farm to the city market.
3. **Market Fees:** Any additional charges levied by the market or transporters.
4. **Total Quantity:** The total weight of tomatoes sold (currently 800 kg).

**What Can Be Calculated**
* **Gross Revenue:** The total income generated at the city market.
  * Calculation: 800 kg × ₹25/kg = **₹20,000**

**What Cannot Be Calculated**
* **Net Profit:** Because the cost of goods sold (purchase price) and the expenses (transportation and fees) are unknown, the final profit cannot be determined.
* **Profit Per Kilogram:** Without the purchase price, the profit margin per kg cann

Test 4 – Consistency and mathematical reasoning

In [83]:

test_prompt = """A Women's Self-Help Group has 12 members. Each member produces 25 handmade bags. Each bag is sold for ₹80. The total production expenses for the entire group are ₹8,000.

Calculate:
1. Total bags produced.
2. Total sales revenue.
3. Total group expenses.
4. Net group profit.
5. Equal profit share per member.
6. Revenue per member.
7. Expense share per member.

Clearly distinguish revenue, expenses, profit, and individual profit share. Do not add expenses to profit."""

messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


Here is the step-by-step calculation for the Women's Self-Help Group based on the data provided.

### 1. Total bags produced
Since there are 12 members and each produces 25 bags:
$$12 \text{ members} \times 25 \text{ bags/member} = 300 \text{ bags}$$

### 2. Total sales revenue
The total revenue is calculated by multiplying the total number of bags produced by the selling price per bag:
$$300 \text{ bags} \times ₹80 \text{ per bag} = ₹24,000$$

### 3. Total group expenses
This value is given directly in the problem statement:
$$\text{Total Expenses} = ₹8,000$$

### 4. Net group profit
Profit is calculated by subtracting total expenses from total revenue:
$$\text{Net Profit} = \text{Total Revenue} - \text{Total Expenses}$$
$$₹24,000 - ₹8,000 = ₹16,000$$

### 5. Equal profit share per member
The total net profit is distributed equally among the 12 members:
$$₹16,000 \div 12 \text{ members} = ₹1,333.33 \text{ (rounded to two decimal places)}$$

### 6. Revenue per member
The total revenue 

Test 5- Transport Cost Optimization

In [84]:

test_prompt = """A rural farmer needs to transport 1,500 kg of vegetables to a wholesale market.

Two transportation options are available:

Option A:
- Small vehicle capacity: 500 kg
- Cost per trip: ₹1,200

Option B:
- Large vehicle capacity: 1,000 kg
- Cost per trip: ₹2,000

Calculate:
1. Number of trips required for each option.
2. Total transportation cost for each option.
3. Which option is cheaper?
4. Total money saved by choosing the cheaper option.

Assume vehicles can make the required trips and partial loads are allowed. Show all calculations and do not exceed vehicle capacity."""

messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=500, do_sample=False)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


### 1. Number of Trips Required

To determine the number of trips, we divide the total weight of the vegetables by the capacity of each vehicle option.

**Option A: Small Vehicle**
*   **Total Weight:** 1,500 kg
*   **Capacity:** 500 kg
*   **Calculation:** $1,500 \div 500 = 3$
*   **Trips Required:** **3 trips**

**Option B: Large Vehicle**
*   **Total Weight:** 1,500 kg
*   **Capacity:** 1,000 kg
*   **Calculation:** $1,500 \div 1,000 = 1.5$
*   **Trips Required:** Since partial loads are allowed, the farmer must make 2 trips to ensure all goods are transported (1 full trip + 1 partial trip).
*   **Trips Required:** **2 trips**

---

### 2. Total Transportation Cost

Next, we calculate the total cost for each option by multiplying the number of trips by the cost per trip.

**Option A: Small Vehicle**
*   **Trips:** 3
*   **Cost per Trip:** ₹1,200
*   **Calculation:** $3 \times ₹1,200 = ₹3,600$
*   **Total Cost:** **₹3,600**

**Option B: Large Vehicle**
*   **Trips:** 2
*   **Cost per

Test 6- Artisan Marketplace Financial Reasoning

In [85]:

test_prompt = """A rural artisan group produces 150 handmade baskets.

Option A:
- Sell all baskets to a wholesaler at ₹120 per basket.
- No additional expenses are specified.

Option B:
- Sell all baskets directly through an online marketplace at ₹180 per basket.
- Marketplace commission: 10% of sales revenue.
- Packaging and shipping expense: ₹1,500 total.

Calculate:
1. Gross revenue for both options.
2. Commission for Option B.
3. Net amount for each option using only the stated costs.
4. Which option gives the higher net amount?
5. Difference between the two options.

Important: Do not invent or assume additional expenses for Option A. Clearly state if any costs are unspecified."""

messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=500, do_sample=False)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


### Financial Analysis of Basket Sales Options

Below is the detailed financial breakdown for the two options provided. Per the instructions, no additional expenses are assumed for Option A beyond the stated price.

#### 1. Gross Revenue Calculation
*   **Total Production:** 150 baskets
*   **Option A:** 150 baskets × ₹120 = **₹18,000**
*   **Option B:** 150 baskets × ₹180 = **₹27,000**

#### 2. Commission for Option B
*   **Total Sales Revenue:** ₹27,000
*   **Commission Rate:** 10%
*   **Calculation:** ₹27,000 × 0.10 = **₹2,700**

#### 3. Net Amount Calculation
*Note: Option A has no specified costs other than the selling price.*

*   **Option A:**
    *   Gross Revenue: ₹18,000
    *   Less Costs: ₹0 (No additional expenses specified)
    *   **Net Amount:** **₹18,000**

*   **Option B:**
    *   Gross Revenue: ₹27,000
    *   Less Commission: ₹2,700
    *   Less Packaging & Shipping: ₹1,500
    *   **Net Amount:** ₹27,000 - ₹2,700 - ₹1,500 = **₹22,800**

#### 4. Comparison: Which O

Test 7 – Rural Youth Employment Analytics

In [86]:

test_prompt = """A rural skill development program enrolled 200 young people.

- 160 participants successfully completed the training.
- 120 of the trained participants obtained employment.
- Each employed participant earns ₹15,000 per month.

Calculate:
1. Training completion rate as a percentage of enrolled participants.
2. Employment rate as a percentage of trained participants.
3. Overall employment rate as a percentage of enrolled participants.
4. Number of trained participants who remain unemployed.
5. Total monthly income generated by employed participants.
6. Average monthly income per enrolled participant, counting unemployed participants as earning zero.

Show formulas and calculations. Clearly distinguish the employment rate among trained participants from the overall employment rate."""

messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


### 1. Training Completion Rate
This measures the success of the program in delivering training to the enrolled cohort.

**Formula:**
$$ \text{Training Completion Rate} = \left( \frac{\text{Number of Successfully Completed Training}}{\text{Total Enrolled Participants}} \right) \times 100 $$

**Calculation:**
$$ \text{Training Completion Rate} = \left( \frac{160}{200} \right) \times 100 = 80\% $$

---

### 2. Employment Rate (Trained Participants)
This measures the success of the program in converting trained individuals into employed individuals.

**Formula:**
$$ \text{Employment Rate (Trained)} = \left( \frac{\text{Number of Employed Trained Participants}}{\text{Number of Trained Participants}} \right) \times 100 $$

**Calculation:**
$$ \text{Employment Rate (Trained)} = \left( \frac{120}{120} \right) \times 100 = 100\% $$

---

### 3. Overall Employment Rate (Enrolled Participants)
This measures the total success rate of the program in securing employment for the entire group, includ